In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df=pd.read_csv('/content/drive/MyDrive/CrimePredictionSystem/CrimeData_Final.csv')

In [ ]:
#Converts date to a proper datetime object
df['date']= pd.to_datetime(df['date'])

In [ ]:
df.head()

In [ ]:
print("Rows, Columns =",df.shape)


In [ ]:
df.columns.tolist()

In [ ]:
df.info()

In [ ]:
df.describe(include="all")

##Identifying Unique values in each colummn

In [ ]:
categorical_cols =  ['crime',
 'location',
 'date',
 'sex',
 'victim_ethnicity',
 'victim_age',
 'time',
 'weather',
 'latitude',
 'longitude',
 'holiday_name',
 'is_holiday',
 'land_use_type',
 'gn_division',
 'gn_pcode',
 'gn_population',
 'gn_distance_m',
 'victim_ethnicity ',
 'DS_Division',
 'Avg_Household_Income',
 'Unemployment_Rate',
 'Building_Density',
 'Road_Density',
 'Land_Area_Density']
print("Unique values")
for col in categorical_cols:
  print(f"{col} : ",df[col].unique())
  # print(f"{col} : ",df[col].nunique())

In [ ]:
print("Unique values")
for col in categorical_cols:
  # print(f"{col} : ",df[col].unique())
  print(f"{col} : ",df[col].nunique())

##Aggregate to : GN Division & Time Window

In [ ]:
#Create weekly time windows
df['week'] = df['date'].dt.to_period('W').apply(lambda r: r.start_time)



In [ ]:
#Aggregate to GN Division, Time Window


aggregated_df = (
    df.groupby(['gn_division', 'week'])
      .agg(
          #crime counts (targets)
          total_crimes=('crime', 'count'),

          burglary_count=('crime', lambda x: x.str.contains('burglary', case=False).sum()),
          theft_count=('crime', lambda x: (x == 'theft').sum()),
          vehicle_count=('crime', lambda x: x.str.contains('vehicle', case=False).sum()),
          robbery_count=('crime', lambda x: x.str.contains('robbery', case=False).sum()),
          drugs_count=('crime', lambda x: x.str.contains('drugs', case=False).sum()),
          stabbing_count=('crime', lambda x: x.str.contains('stabbing', case=False).sum()),

          #demographic context (ethical aggregation)
          avg_victim_age=('victim_age', 'mean'),
          female_victim_ratio=('sex', lambda x: (x == 'f').mean()),

          #spacial/environmental context - GN-level spatial exposure variables
          population=('gn_population', 'first'), #Population = exposure size
          mean_distance_m=('gn_distance_m', 'mean'), #Distance = accessibility / remoteness
          urban_ratio=('land_use_type', lambda x: (x == 'General Urban').mean()), # Urban ratio = land-use intensity

          #Temporal context
          #Proportion of days in the week that were: holidays, rainy
          holiday_ratio=('is_holiday', 'mean'),
          rainy_ratio=('weather', lambda x: (x.str.lower() == 'rainy').mean()),

          ds_division=('DS_Division', 'first')
      )
      .reset_index()
)

##Create complete GN×Week grid (zero-crime weeks)

In [ ]:
all_gn = aggregated_df['gn_division'].unique()
all_weeks = pd.date_range(
    aggregated_df['week'].min(),
    aggregated_df['week'].max(),
    freq='W-MON'
)

full_index = pd.MultiIndex.from_product(
    [all_gn, all_weeks],
    names=['gn_division', 'week']
)

completed_df = (
    full_index.to_frame(index=False)
    .merge(aggregated_df, on=['gn_division', 'week'], how='left')
)

crime_cols = [
    'total_crimes','burglary_count','theft_count',
    'vehicle_count','robbery_count','drugs_count','stabbing_count'
]

completed_df[crime_cols] = completed_df[crime_cols].fillna(0)


completed_df['avg_victim_age'] = (
    completed_df.groupby('gn_division')['avg_victim_age']
                .transform(lambda x: x.fillna(x.median()))
)


ratio_cols = ['female_victim_ratio','urban_ratio','holiday_ratio','rainy_ratio']
completed_df[ratio_cols] = completed_df[ratio_cols].fillna(0)

# Enure Every GN-week row gets a valid population and distance value
completed_df[['population','mean_distance_m']] = (
    completed_df.groupby('gn_division')[['population','mean_distance_m']]
                .ffill().bfill()
)


##Integrate demographic & environmental variables

In [ ]:
context_cols = [
    'Unemployment_Rate',
    'Building_Density',
    'Avg_Household_Income',
    'Land_Area_Density',
    'Road_Density'
]

for col in context_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

gn_context = (
    df.groupby('gn_division')
      .agg(
          unemployment_rate=('Unemployment_Rate', 'mean'),
          avg_income=('Avg_Household_Income', 'mean'),
          building_density=('Building_Density', 'mean'),
          land_area_density=('Land_Area_Density', 'mean'),
          road_density=('Road_Density', 'mean'),
          ds_division=('DS_Division', 'first')
      )
      .reset_index()
)

completed_df = completed_df.merge(
    gn_context,
    on='gn_division',
    how='left'
)


##Feature engineering (final feature table)

In [ ]:
df_features = completed_df.copy()

# Time features
df_features['year'] = df_features['week'].dt.year
df_features['week_of_year'] = df_features['week'].dt.isocalendar().week.astype(int)

df_features['week_sin'] = np.sin(2*np.pi*df_features['week_of_year']/52)
df_features['week_cos'] = np.cos(2*np.pi*df_features['week_of_year']/52)

# Scale-friendly transforms
df_features['log_population'] = np.log1p(df_features['population'])
df_features['distance_km'] = df_features['mean_distance_m'] / 1000

df_features.drop(
    columns=['week','week_of_year','population','mean_distance_m'],
    inplace=True
)

df_features[crime_cols] = df_features[crime_cols].astype(int)


##Dependent variable management (clean & final)

In [ ]:
df_features.rename(columns={'ds_division_y': 'ds_division'}, inplace=True)

df_features.drop(columns=['ds_division_x'], inplace=True)

df_features['ds_division'].isna().sum()


In [ ]:
df_features.to_csv("df_features.csv", index=False)

from google.colab import files
files.download("df_features.csv")


In [ ]:
df_model = df_features.copy()

In [ ]:
crime_types = {
    'burglary':'y_burglary',
    'theft':'y_theft',
    'vehicle':'y_vehicle',
    'robbery':'y_robbery',
    'drugs':'y_drugs',
    'stabbing':'y_stabbing'
}

In [ ]:
# Create binary targets

df_model['y_burglary'] = (df_model['burglary_count'] > 0).astype(int)
df_model['y_theft'] = (df_model['theft_count'] > 0).astype(int)
df_model['y_vehicle'] = (df_model['vehicle_count'] > 0).astype(int)
df_model['y_robbery'] = (df_model['robbery_count'] > 0).astype(int)
df_model['y_drugs'] = (df_model['drugs_count'] > 0).astype(int)
df_model['y_stabbing'] = (df_model['stabbing_count'] > 0).astype(int)


In [ ]:
leakage_cols = [
    # Spatial identifiers (NOT causal features)
    'gn_division',
    'ds_division',

    # Direct outcome leakage
    'total_crimes',

    # Crime-type counts (post-event information)
    'burglary_count',
    'theft_count',
    'vehicle_count',
    'robbery_count',
    'drugs_count',
    'stabbing_count',

    # Binary targets (never in X)
    'y_burglary',
    'y_theft',
    'y_vehicle',
    'y_robbery',
    'y_drugs',
    'y_stabbing'
]



In [ ]:
#Time-aware train test split

In [ ]:
df_model['week'] = completed_df['week'].values

# sort by time
df_model = df_model.sort_values(by='week').reset_index(drop=True)

#define X and y for each crime type
target_cols = [
    'y_burglary',
    'y_theft',
    'y_vehicle',
    'y_robbery',
    'y_drugs',
    'y_stabbing'
]

X = df_model.drop(columns=leakage_cols + ['week'])

for col in target_cols:
  target_col = col
  y = df_model[target_col]

  # perform the chronological split
  split_index = int(len(df_model)* 0.8)

  X_train = X.iloc[:split_index]
  X_test = X.iloc[split_index:]

  y_train = y.iloc[:split_index]
  y_test = y.iloc[split_index:]


  print("Train size: ",X_train.shape)
  print("Test size: ",X_test.shape)

  print("\nTarget distribution (train):")
  print(y_train.value_counts(normalize=True))

  print("\nTarget distribution (test):")
  print(y_test.value_counts(normalize=True))


In [ ]:
# Scaling and Normalization

In [ ]:
from sklearn.preprocessing import StandardScaler

scale_cols = ['avg_victim_age','unemployment_rate','avg_income','building_density','land_area_density','road_density','log_population','distance_km','year']

scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test_scaled[scale_cols] = scaler.transform(X_test[scale_cols])

X_train_scaled[scale_cols].describe().loc[['mean','std']]

In [ ]:
X_test_scaled[scale_cols].describe().loc[['mean','std']]